# Lab 05: CNN Fundamentals and Feature Maps

            **Duration:** 3 hours  
            **Lecture alignment:** Week 5 — Convolutional neural networks  
            **CLO mapping:** CLO-1, CLO-2, CLO-3  
            **Framework:** PyTorch (standalone, credential-free, CPU smoke-test with optional GPU)

            ## Learning objectives

            - Calculate convolution/pooling shapes and receptive fields.
- Train a compact CNN on generated images.
- Compare CNN and MLP inductive biases using accuracy, parameters, and latency.

            ## Three-hour activity plan

            - 0–30 min: image generation and kernel geometry
- 30–70 min: padding, stride, pooling, and receptive fields
- 70–130 min: CNN implementation and training
- 130–160 min: CNN/MLP comparison and feature maps
- 160–180 min: checks and interpretation


## Book grounding

            - Goodfellow, Bengio, and Courville, *Deep Learning*, MIT Press, 2016.
- Zhang, Lipton, Li, and Smola, *Dive into Deep Learning*, Cambridge University Press, 2024.
- Prince, *Understanding Deep Learning*, MIT Press, 2023.

            The notebook paraphrases concepts and supplies original code; it does not reproduce book text.


In [ ]:
from pathlib import Path
import json, math, os, random, time
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset

FAST_MODE = True
RUN_EXTENSION = False
SEED = 20265
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(min(2, os.cpu_count() or 1))

if Path("/content").exists():
    ARTIFACT_DIR = Path("/content/artifacts/lab_05")
else:
    ARTIFACT_DIR = Path.cwd() / "tmp" / "course_build" / "runtime" / "lab_05"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print({"lab": 5, "device": str(DEVICE), "fast_mode": FAST_MODE,
       "artifacts": str(ARTIFACT_DIR), "torch": torch.__version__})


## Predict before running

Will the CNN use fewer parameters than the MLP while matching its accuracy? Explain which architectural assumption supports your prediction.

Record a brief prediction in your own words before executing the experiment, then revisit it in the exit reflection.


## Activity 1 — Generate images and inspect convolution geometry


In [ ]:
def make_line_images(n, size=16):
    labels = torch.randint(0, 2, (n,))
    images = .08 * torch.randn(n, 1, size, size)
    for i, label in enumerate(labels):
        if label.item() == 0:
            col = int(torch.randint(3, size-3, (1,)).item()); images[i, 0, 2:-2, col-1:col+2] += 1
        else:
            row = int(torch.randint(3, size-3, (1,)).item()); images[i, 0, row-1:row+2, 2:-2] += 1
    return images.clamp(0, 1), labels

X, y = make_line_images(700 if FAST_MODE else 3000)
split = int(.8*len(X)); Xtr, Xte, ytr, yte = X[:split], X[split:], y[:split], y[split:]
probe = nn.Conv2d(1, 4, kernel_size=3, stride=1, padding=1)
assert probe(X[:2]).shape == (2, 4, 16, 16)
# Two 3x3 convolutions with a 2x2 pool between them give an effective receptive field of 8.
receptive_field = 1
jump = 1
for kernel, stride in [(3,1), (2,2), (3,1)]:
    receptive_field += (kernel-1)*jump; jump *= stride
print({"receptive_field": receptive_field, "output_jump": jump})


## Activity 2 — Train a CNN and a parameter-matched MLP


In [ ]:
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 6, 3, padding=1)
        self.conv2 = nn.Conv2d(6, 10, 3, padding=1)
        self.head = nn.Linear(10, 2)
    def features(self, x):
        x = F.max_pool2d(F.relu(self.conv1(x)), 2)
        return F.relu(self.conv2(x))
    def forward(self, x):
        x = self.features(x); x = F.adaptive_avg_pool2d(x, 1).flatten(1)
        return self.head(x)

class SmallMLP(nn.Module):
    def __init__(self):
        super().__init__(); self.net = nn.Sequential(nn.Flatten(), nn.Linear(256, 32), nn.ReLU(), nn.Linear(32, 2))
    def forward(self, x): return self.net(x)

def fit_model(model, epochs=6):
    model = model.to(DEVICE); opt = torch.optim.Adam(model.parameters(), lr=.015)
    loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=64, shuffle=True,
                        generator=torch.Generator().manual_seed(SEED))
    history = []
    for _ in range(epochs):
        model.train(); total = 0
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE); opt.zero_grad()
            loss = F.cross_entropy(model(xb), yb); loss.backward(); opt.step(); total += loss.item()*len(xb)
        history.append(total/len(Xtr))
    model.eval(); start = time.perf_counter()
    with torch.no_grad(): logits = model(Xte.to(DEVICE));
    latency = (time.perf_counter()-start)/len(Xte)
    acc = (logits.argmax(1).cpu()==yte).float().mean().item()
    return model, history, acc, latency

cnn_result = fit_model(SmallCNN(), 5 if FAST_MODE else 15)
mlp_result = fit_model(SmallMLP(), 5 if FAST_MODE else 15)
comparison = {
    "CNN": {"accuracy": cnn_result[2], "parameters": sum(p.numel() for p in cnn_result[0].parameters()), "seconds_per_item": cnn_result[3]},
    "MLP": {"accuracy": mlp_result[2], "parameters": sum(p.numel() for p in mlp_result[0].parameters()), "seconds_per_item": mlp_result[3]},
}
print(json.dumps(comparison, indent=2))


## Activity 3 — Feature-map visualization


In [ ]:
cnn = cnn_result[0]
with torch.no_grad(): fmap = cnn.features(Xte[:1].to(DEVICE)).cpu()[0]
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for channel, ax in enumerate(axes.flat):
    ax.imshow(fmap[channel], cmap="viridis"); ax.axis("off"); ax.set_title(f"map {channel}")
fig.suptitle("Second convolution feature maps"); fig.tight_layout()
fig.savefig(ARTIFACT_DIR / "feature_maps.png", dpi=150); plt.show()
(ARTIFACT_DIR / "comparison.json").write_text(json.dumps(comparison, indent=2))


## Automated checks


In [ ]:
assert receptive_field == 8
assert cnn_result[2] > .85 and mlp_result[2] > .75
assert fmap.shape == (10, 8, 8)
assert (ARTIFACT_DIR / "feature_maps.png").exists()
print("All Lab 05 checks passed.")


## Deliverables

                - Shape and receptive-field calculations
- Trained CNN/MLP comparison JSON
- Feature-map visualization and architectural interpretation

                Submit the executed notebook and the files created in `/content/artifacts/lab_05/`.


## Disabled extension

The following challenge is intentionally disabled by default so the CPU baseline stays quick.


In [ ]:
if RUN_EXTENSION:
    print("Extension: replace max pooling with average pooling and add dilation to the second convolution.")
else:
    print("Extension disabled: compare pooling choices and dilated receptive fields.")


## Exit reflection

In 4–6 sentences, state: (1) whether your prediction was supported, (2) the strongest evidence,
(3) one failure mode or limitation, and (4) the next experiment you would run. Include at least
one measured value rather than only a general claim.
